In [3]:
import polars as pl

# Load the CSV file
df = pl.read_csv("./fresh-grad-filter.csv")

# Define the schema for the educations JSON structure
# Based on your final_structure.json, educations is a list of objects
education_schema = pl.List(
    pl.Struct([
        pl.Field("degree_user_input", pl.String),
        pl.Field("degree_standardized", pl.String),
        pl.Field("university", pl.String),
        pl.Field("major_user_input", pl.String),
        pl.Field("major_standardized", pl.String),
        pl.Field("description", pl.String),
        pl.Field("start_date", pl.String),
        pl.Field("end_date", pl.String),
    ])
)

# Parse JSON strings to structs, then extract university
df_with_universities = (
    df
    # Parse the JSON string in 'educations' column with explicit dtype
    .with_columns(
        pl.col("educations").str.json_decode(dtype=education_schema, infer_schema_length=1000)
    )
    # Explode the list to get one row per education entry
    .explode("educations")
    # Extract the university field from each education struct
    .with_columns(
        pl.col("educations").struct.field("university").alias("university")
    )
    # Filter out null universities (though presence_pct is 100%)
    .filter(pl.col("university").is_not_null())
)

# Count frequencies and get top 100 most popular universities
top_100_universities = (
    df_with_universities
    .group_by("university")
    .agg(pl.len().alias("frequency"))
    .sort("frequency", descending=True)
    .limit(100)
)

# Display the results
print("Top 100 Most Popular Universities:")
print("=" * 70)
for i, row in enumerate(top_100_universities.rows(), 1):
    uni_name = row[0] if len(row[0]) <= 55 else row[0][:52] + "..."
    print(f"{i:3d}. {uni_name:55s} - {row[1]:>8,} occurrences")

# Optionally save to CSV
top_100_universities.write_csv("top_100_universities.csv")

# Show summary statistics
print("\n" + "=" * 70)
print(f"Total unique universities: {df_with_universities['university'].n_unique():,}")
print(f"Total education entries processed: {len(df_with_universities):,}")

Top 100 Most Popular Universities:
  1. RMIT University Vietnam                                 -    2,093 occurrences
  2. RMIT University                                         -      335 occurrences
  3. British University Vietnam                              -      272 occurrences
  4. Fulbright University Vietnam                            -      248 occurrences
  5. Swinburne University of Technology                      -      229 occurrences
  6. VinUniversity                                           -      224 occurrences
  7. Staffordshire University                                -       89 occurrences
  8. Hanoi-Amsterdam High School for the Gifted              -       41 occurrences
  9. Tran Dai Nghia Specialized High School                  -       37 occurrences
 10. VNU-HCM High School for the Gifted                      -       31 occurrences
 11. Foreign Language Specialized School                     -       31 occurrences
 12. Le Hong Phong High School For The Gi

/tmp/ipykernel_160265/1321417228.py:26: DeprecationWarning: `Expr.str.json_decode` with `infer_schema_length` is deprecated and has no effect on execution.
  pl.col("educations").str.json_decode(dtype=education_schema, infer_schema_length=1000)


In [9]:
import polars as pl

# 1. Load the dataset
df = pl.read_csv("../Data/final.csv")


# 3. Count occurrences for an array of university names
target_universities = ["BUV", "British University Vietnam", "Swinburne", "FUV", "Fulbright University Vietnam", "RMIT","Royal Melbourne Institute of Technology Vietnam","VinUniversity","VIN"]

# Parse the JSON strings, explode the list into individual rows, and extract the university field
df_unis = (
    df.select(pl.col("educations").str.json_decode(edu_schema))
    .explode("educations")
    .select(pl.col("educations").struct.field("university").str.to_lowercase().alias("uni_name"))
    .filter(pl.col("uni_name").is_not_null())
)

# Compute the case-insensitive substring match count for each name in the array
university_counts = {
    uni: df_unis.filter(pl.col("uni_name").str.contains(uni.lower())).height
    for uni in target_universities
}

print("University Occurrences:", university_counts)

University Occurrences: {'BUV': 29, 'British University Vietnam': 841, 'Swinburne': 1149, 'FUV': 1, 'Fulbright University Vietnam': 642, 'RMIT': 15165, 'Royal Melbourne Institute of Technology Vietnam': 3, 'VinUniversity': 353, 'VIN': 3963}


In [2]:
import polars as pl

# 1. Load the dataset
df = pl.read_csv("../../Data/final.csv")

# Update schema to include degree_standardized
edu_schema = pl.List(pl.Struct({
    "university": pl.String,
    "degree_standardized": pl.String,
    "end_date": pl.String
}))

# Target universities list
target_universities = [
    "BUV", "British University Vietnam", "Swinburne", "FUV", 
    "Fulbright University Vietnam", "RMIT", "Royal Melbourne Institute of Technology Vietnam",
    "VinUniversity"
]
pattern = "|".join([uni.lower() for uni in target_universities])

# 2. Extract unique row IDs matching both university and bachelor degree criteria
matching_ids = (
    df.select(["id", pl.col("educations").str.json_decode(edu_schema)])
    .explode("educations")
    .filter(
        (pl.col("educations").struct.field("university").str.to_lowercase().str.contains(pattern)) &
        (pl.col("educations").struct.field("degree_standardized") == "bachelor") &
        (pl.col("educations").struct.field("end_date").is_in(["2027-01-01", "2026-01-01", "2025-01-01", "2024-01-01"]))
    )
    .select("id")
    .unique()
)

# 3. Filter the original DataFrame and export to CSV
filtered_df = df.join(matching_ids, on="id", how="inner")
filtered_df.write_csv("fresh-grad-filter.csv")